# SIT720 Machine Learning - Task 2.1P: Data Wrangling

**Student:** Zenith Anthony Dabarera  
**Student ID:** s225534396  
**Dataset:** City of Melbourne, *Microclimate sensors data*

This notebook answers all seven problem-solving questions in Task 2.1P. It is designed to work with either:

1. a local file named `microclimate-sensors-data.csv`, or
2. the current CSV export from the City of Melbourne Open Data Portal.

> **Before submission:** Run **Kernel > Restart & Run All**, check every output and graph, then save the notebook with outputs. The City dataset is live, so the exact row count and the answers to Questions 6 and 7 can change when the portal is refreshed.

## 0. Imports and display settings

The main libraries are:

- **pandas** for loading, cleaning, grouping and summary statistics;
- **NumPy** for numerical operations;
- **Matplotlib** for plots; and
- **scikit-learn** for min-max scaling.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

OFFICIAL_CSV_URLS = [
    "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/microclimate-sensors-data/exports/csv",
    "https://data.melbourne.vic.gov.au/api/v2/catalog/datasets/microclimate-sensors-data/exports/csv?delimiter=%2C",
]

LOCAL_CANDIDATES = [
    Path("microclimate-sensors-data.csv"),
    Path("data/microclimate-sensors-data.csv"),
    Path("/mnt/data/microclimate-sensors-data.csv"),
]

## 1. Load the microclimate data

The function first searches for a local CSV. If it is not found, it attempts the official online export. This makes the notebook reproducible while still allowing it to run in environments where direct internet access is blocked.

In [ ]:
def read_microclimate_data():
    # Load the dataset from a local CSV or the official City of Melbourne export.
    load_errors = []

    for path in LOCAL_CANDIDATES:
        if path.exists():
            try:
                frame = pd.read_csv(path, low_memory=False)
                print(f"Loaded local file: {path.resolve()}")
                return frame
            except Exception as exc:
                load_errors.append(f"{path}: {exc}")

    for url in OFFICIAL_CSV_URLS:
        try:
            frame = pd.read_csv(url, low_memory=False)
            print(f"Loaded official City of Melbourne export: {url}")
            return frame
        except Exception as exc:
            load_errors.append(f"{url}: {exc}")

    details = "\n".join(f"- {item}" for item in load_errors)
    raise FileNotFoundError(
        "The dataset could not be loaded. Download the CSV from the City of Melbourne "
        "Open Data Portal, rename it 'microclimate-sensors-data.csv', and place it in "
        "the same folder as this notebook. Attempts made:\n" + details
    )

raw_input = read_microclimate_data()
print(f"Dataset shape: {raw_input.shape[0]:,} rows x {raw_input.shape[1]} columns")
display(raw_input.head())

### Standardise column names and data types

The portal has used slightly different labels in different exports, such as `received_at` instead of `Time`. The mapping below converts supported variants to one consistent set of names.

In [ ]:
def simplified_name(name):
    return "".join(ch.lower() for ch in str(name) if ch.isalnum())

canonical_by_key = {
    "deviceid": "Device_id",
    "time": "Time",
    "receivedat": "Time",
    "sensorlocation": "SensorLocation",
    "latlong": "LatLong",
    "minimumwinddirection": "MinimumWindDirection",
    "averagewinddirection": "AverageWindDirection",
    "maximumwinddirection": "MaximumWindDirection",
    "minimumwindspeed": "MinimumWindSpeed",
    "averagewindspeed": "AverageWindSpeed",
    "gustwindspeed": "GustWindSpeed",
    "airtemperature": "AirTemperature",
    "relativehumidity": "RelativeHumidity",
    "atmosphericpressure": "AtmosphericPressure",
    "pm25": "PM25",
    "pm10": "PM10",
    "noise": "Noise",
}

rename_map = {}
for original in raw_input.columns:
    key = simplified_name(original)
    if key in canonical_by_key:
        rename_map[original] = canonical_by_key[key]

raw_df = raw_input.rename(columns=rename_map).copy()
required_columns = list(canonical_by_key.values())
missing_columns = [col for col in required_columns if col not in raw_df.columns]
if missing_columns:
    raise KeyError(f"Required columns are missing after renaming: {missing_columns}")

raw_df = raw_df[required_columns]

numeric_features = [
    "MinimumWindDirection", "AverageWindDirection", "MaximumWindDirection",
    "MinimumWindSpeed", "AverageWindSpeed", "GustWindSpeed",
    "AirTemperature", "RelativeHumidity", "AtmosphericPressure",
    "PM25", "PM10", "Noise",
]
for col in numeric_features:
    raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

print("Standardised columns:")
print(raw_df.columns.tolist())
print("\nData types:")
display(raw_df.dtypes.to_frame("dtype"))

## Question 1 - Feature names and missing entries

`isna().sum()` counts the missing entries in each feature. The percentage is also shown because a raw count is easier to interpret when compared with the total number of rows.

In [ ]:
missing_summary = pd.DataFrame({
    "Missing entries": raw_df.isna().sum(),
    "Missing percentage": raw_df.isna().mean().mul(100),
}).sort_values("Missing entries", ascending=False)

print("Missing entries by feature:")
display(missing_summary)

## Question 2 - Fill missing entries

### Choice of replacement statistic

For the numerical sensor features, this notebook uses the **median** of the observed values. Sensor measurements such as PM2.5, PM10, wind speed and noise can contain spikes and strongly skewed distributions. The median is resistant to extreme values, whereas the mean can be pulled toward a few unusually large readings.

`SensorLocation` and `LatLong` are not numerical measurements, so a mean or median would have no meaning. They describe device metadata. Their missing values are therefore filled with the most common observed value for the same `Device_id`, followed by the overall mode only when a device-level value is unavailable.

In [ ]:
df = raw_df.copy()

# Numerical features: median imputation.
replacement_rows = []
for col in numeric_features:
    missing_before = int(df[col].isna().sum())
    replacement = float(df[col].median())
    if missing_before:
        df[col] = df[col].fillna(replacement)
    replacement_rows.append({
        "Feature": col,
        "Missing before": missing_before,
        "Method": "Median",
        "Replacement value": replacement,
    })

replacement_table = pd.DataFrame(replacement_rows)
print("Numerical replacement values:")
display(replacement_table)

# Categorical/device metadata: device-wise mode, then global mode.
def fill_from_device_mode(frame, column):
    output = frame[column].copy()
    device_mode = (
        frame.dropna(subset=[column])
             .groupby("Device_id")[column]
             .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
    )
    missing_mask = output.isna()
    output.loc[missing_mask] = frame.loc[missing_mask, "Device_id"].map(device_mode)
    if output.isna().any():
        global_mode = frame[column].mode(dropna=True)
        fallback = global_mode.iloc[0] if not global_mode.empty else "Unknown"
        output = output.fillna(fallback)
    return output

for metadata_col in ["SensorLocation", "LatLong"]:
    before = int(df[metadata_col].isna().sum())
    df[metadata_col] = fill_from_device_mode(df, metadata_col)
    print(f"{metadata_col}: filled {before:,} missing values using device-wise mode and global fallback.")

print("\nRemaining missing values after imputation:")
display(df.isna().sum().to_frame("Remaining missing"))

## Question 3 - PM25 and PM10 distributions and correlation

Two views are useful:

1. the full histogram, which retains all extreme observations; and
2. a histogram limited to the 99th percentile, which makes the main body of each distribution visible.

Both Pearson and Spearman correlations are calculated. Pearson measures linear association and is sensitive to outliers. Spearman measures whether the variables generally increase together based on their ranks and is more robust for skewed data.

In [ ]:
pm = df[["PM25", "PM10"]].copy()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for j, col in enumerate(["PM25", "PM10"]):
    axes[0, j].hist(pm[col].dropna(), bins=60, edgecolor="black", linewidth=0.3)
    axes[0, j].set_title(f"{col}: full distribution")
    axes[0, j].set_xlabel(col)
    axes[0, j].set_ylabel("Frequency")

    upper = pm[col].quantile(0.99)
    axes[1, j].hist(pm.loc[pm[col] <= upper, col].dropna(), bins=60, edgecolor="black", linewidth=0.3)
    axes[1, j].set_title(f"{col}: values up to 99th percentile ({upper:.2f})")
    axes[1, j].set_xlabel(col)
    axes[1, j].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

pm_diagnostics = pd.DataFrame({
    "Mean": pm.mean(),
    "Median": pm.median(),
    "Skewness": pm.skew(),
    "Zero percentage": pm.eq(0).mean().mul(100),
    "99th percentile": pm.quantile(0.99),
    "Maximum": pm.max(),
})
print("PM distribution diagnostics:")
display(pm_diagnostics)

pearson_r = pm.corr(method="pearson").loc["PM25", "PM10"]
spearman_rho = pm.corr(method="spearman").loc["PM25", "PM10"]
print(f"Pearson correlation (PM25 vs PM10):  {pearson_r:.4f}")
print(f"Spearman correlation (PM25 vs PM10): {spearman_rho:.4f}")

plt.figure(figsize=(7, 5))
sample = pm.sample(min(20_000, len(pm)), random_state=720)
plt.scatter(sample["PM25"], sample["PM10"], alpha=0.20, s=8)
plt.xlabel("PM25")
plt.ylabel("PM10")
plt.title("PM25 versus PM10 (random sample for readability)")
plt.tight_layout()
plt.show()

### Interpretation of Question 3

Use the diagnostics printed above in the final explanation:

- A **positive skewness** together with a mean greater than the median indicates a right-skewed distribution.
- Large maximum values compared with the 99th percentile indicate rare extreme readings.
- A positive correlation indicates that PM25 and PM10 tend to rise and fall together, although correlation does not prove that one causes the other.
- If Spearman is noticeably stronger than Pearson, the association is more consistently monotonic than strictly linear and the outliers affect Pearson's coefficient.

## Question 4 - Split and encode `LatLong`

`LatLong` contains two continuous quantities in one string. The appropriate transformation is to split it at the comma and convert the two parts to floating-point columns named `Latitude` and `Longitude`.

This preserves distance and ordering information. Label encoding would incorrectly treat each coordinate pair as an arbitrary category, while one-hot encoding would produce many sparse columns and lose the continuous geographic meaning.

In [ ]:
coordinates = df["LatLong"].astype(str).str.split(",", n=1, expand=True)
df["Latitude"] = pd.to_numeric(coordinates[0].str.strip(), errors="coerce")
df["Longitude"] = pd.to_numeric(coordinates[1].str.strip(), errors="coerce")

# If malformed coordinate strings exist, fill from the device-level median coordinate.
for coord in ["Latitude", "Longitude"]:
    device_median = df.groupby("Device_id")[coord].transform("median")
    df[coord] = df[coord].fillna(device_median).fillna(df[coord].median())

print("Encoded coordinate values:")
display(df[["LatLong", "Latitude", "Longitude"]].head(10))

## Question 5 - Min-max scaling of continuous features

Min-max scaling uses:

\[
x_{scaled}=\frac{x-x_{min}}{x_{max}-x_{min}}
\]

The transformation maps each selected feature to the interval from 0 to 1. It changes the numerical scale but does not change the order of observations or the basic distribution shape. When a feature has extreme outliers, most ordinary values can be compressed into a small part of the 0-1 interval.

In [ ]:
continuous_features = numeric_features + ["Latitude", "Longitude"]

scaler = MinMaxScaler()
scaled_values = scaler.fit_transform(df[continuous_features])
scaled_df = pd.DataFrame(scaled_values, columns=continuous_features, index=df.index)

scale_check = pd.DataFrame({
    "Original min": df[continuous_features].min(),
    "Original max": df[continuous_features].max(),
    "Scaled min": scaled_df.min(),
    "Scaled max": scaled_df.max(),
})
print("Range before and after min-max scaling:")
display(scale_check)

# Plot before and after for all continuous features.
for col in continuous_features:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
    axes[0].hist(df[col].dropna(), bins=50, edgecolor="black", linewidth=0.25)
    axes[0].set_title(f"{col} - before scaling")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Frequency")

    axes[1].hist(scaled_df[col].dropna(), bins=50, edgecolor="black", linewidth=0.25)
    axes[1].set_title(f"{col} - after scaling")
    axes[1].set_xlabel("Scaled value")
    axes[1].set_ylabel("Frequency")

    plt.tight_layout()
    plt.show()

### Interpretation of Question 5

The before-and-after histograms should have the same relative pattern because min-max scaling is a linear transformation. Only the axis changes. The minimum becomes 0 and the maximum becomes 1. Differences that appear visually usually result from axis labelling or from extreme observations compressing the majority of values.

**Modelling note:** wind direction is circular: 359 degrees is close to 0 degrees. For a predictive model, sine and cosine encoding would normally be better than directly scaling direction. It is retained here because the task specifically asks for min-max scaling of continuous features.

## Question 6 - Which area has the biggest noise?

"Biggest noise" is interpreted as the area with the **highest mean observed noise level**, because one isolated maximum may be a temporary spike or sensor anomaly. Median, maximum and observation count are also reported to support the conclusion.

The calculation uses the original observed noise values rather than the imputed values, so the ranking is not influenced by inserted medians.

In [ ]:
noise_analysis = raw_df[["Device_id", "SensorLocation", "Noise"]].copy()
noise_analysis["SensorLocation"] = fill_from_device_mode(noise_analysis, "SensorLocation")
noise_analysis["Noise"] = pd.to_numeric(noise_analysis["Noise"], errors="coerce")

noise_by_area = (
    noise_analysis.dropna(subset=["SensorLocation", "Noise"])
                  .groupby("SensorLocation")["Noise"]
                  .agg(Mean="mean", Median="median", Maximum="max", Observations="count")
                  .sort_values("Mean", ascending=False)
)

print("Top areas by mean observed noise:")
display(noise_by_area.head(10))

if noise_by_area.empty:
    print("No valid noise/location observations were available.")
else:
    noisiest_area = noise_by_area.index[0]
    noisiest = noise_by_area.iloc[0]
    print(
        f"Answer: {noisiest_area} has the highest mean observed noise "
        f"({noisiest['Mean']:.2f}), based on {int(noisiest['Observations']):,} readings. "
        f"Its median is {noisiest['Median']:.2f} and its maximum is {noisiest['Maximum']:.2f}."
    )

plt.figure(figsize=(10, 5))
noise_by_area["Mean"].sort_values().plot(kind="barh")
plt.xlabel("Mean observed noise")
plt.ylabel("Sensor location")
plt.title("Mean noise by area")
plt.tight_layout()
plt.show()

## Question 7 - In 2025, which month had the most wind?

The timestamp is parsed as UTC and converted to Melbourne local time. The monthly comparison uses the **mean observed `AverageWindSpeed`**, rather than the sum, because months can have different numbers of valid readings. The median, maximum and observation count are also displayed.

In [ ]:
wind_analysis = raw_df[["Time", "AverageWindSpeed"]].copy()
wind_analysis["AverageWindSpeed"] = pd.to_numeric(wind_analysis["AverageWindSpeed"], errors="coerce")

# Parse timezone-aware timestamps and convert them to Melbourne local time.
parsed_utc = pd.to_datetime(wind_analysis["Time"], errors="coerce", utc=True)
try:
    wind_analysis["LocalTime"] = parsed_utc.dt.tz_convert("Australia/Melbourne")
except Exception:
    # Fallback for environments without timezone data.
    wind_analysis["LocalTime"] = pd.to_datetime(
        wind_analysis["Time"].astype(str).str.slice(0, 19), errors="coerce"
    )

wind_2025 = wind_analysis[
    wind_analysis["LocalTime"].dt.year.eq(2025)
].dropna(subset=["LocalTime", "AverageWindSpeed"]).copy()

wind_2025["Month number"] = wind_2025["LocalTime"].dt.month
wind_2025["Month"] = wind_2025["LocalTime"].dt.month_name()

monthly_wind = (
    wind_2025.groupby(["Month number", "Month"])["AverageWindSpeed"]
             .agg(Mean="mean", Median="median", Maximum="max", Observations="count")
             .reset_index()
             .sort_values("Month number")
             .set_index("Month")
)

print("Monthly wind statistics for 2025:")
display(monthly_wind[["Mean", "Median", "Maximum", "Observations"]])

if monthly_wind.empty:
    print("No valid 2025 AverageWindSpeed observations were found. Check the dataset date range.")
else:
    windiest_month = monthly_wind["Mean"].idxmax()
    windiest = monthly_wind.loc[windiest_month]
    print(
        f"Answer: {windiest_month} had the highest mean AverageWindSpeed in 2025 "
        f"({windiest['Mean']:.3f}), based on {int(windiest['Observations']):,} valid readings."
    )

plt.figure(figsize=(10, 4.5))
monthly_wind["Mean"].plot(kind="bar")
plt.ylabel("Mean AverageWindSpeed")
plt.xlabel("Month")
plt.title("Mean average wind speed by month - 2025")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Final automatically generated answer summary

The next cell collects the main results in one place. Copy the exact noisiest-area and windiest-month values into the PDF report after running the notebook on the dataset used for submission.

In [ ]:
print("SIT720 Task 2.1P - Final Results")
print("=" * 45)
print(f"Rows analysed: {len(raw_df):,}")
print(f"Columns analysed: {raw_df.shape[1]}")
print(f"Pearson PM25-PM10 correlation: {pearson_r:.4f}")
print(f"Spearman PM25-PM10 correlation: {spearman_rho:.4f}")

if not noise_by_area.empty:
    print(f"Noisiest area by mean observed Noise: {noisiest_area} ({noisiest['Mean']:.2f})")
else:
    print("Noisiest area: not available")

if not monthly_wind.empty:
    print(f"Windiest month in 2025 by mean AverageWindSpeed: {windiest_month} ({windiest['Mean']:.3f})")
else:
    print("Windiest month in 2025: not available")

## References

- City of Melbourne Open Data Portal. *Microclimate sensors data*.  
  https://data.melbourne.vic.gov.au/explore/dataset/microclimate-sensors-data/
- pandas documentation. *Working with missing data*.  
  https://pandas.pydata.org/docs/user_guide/missing_data.html
- scikit-learn documentation. *MinMaxScaler*.  
  https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html
- Matplotlib documentation. *Histograms*.  
  https://matplotlib.org/stable/gallery/statistics/hist.html